Aim of this project: Train a model to predict unemployment rate in the country. It's a synthetic dataset and has correlation problem which makes it hard for the dataset to learn anything. Still in the process of solving the problem(used standardization and normalization to solve it). The training algorithm used is XGboost cause of its robustness.

In [2]:
#Import Libraries
import pandas as pd
import numpy as np# for numerical computation functions not available in pandas library on the dataframe as it resembles an array.
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
%matplotlib inline

In [3]:
# Loading and reading the file
unemployment = pd.read_csv('Unemployment-Dataset.csv')

In [4]:
# Create a copy of the dataset
unemployment_copy = unemployment.copy()

In [6]:
unemployment #To check duplicates using PyCharm features.

,Year,Geopolitical_Zone,State_Example,Unemployment_Rate_Pct,Underemployment_Rate_Pct,Youth_Unemployment_15_24_Pct,Youth_Unemployment_25_34_Pct,Labor_Force_Participation_Rate_Pct,NEET_Rate_Pct,Informal_Employment_Rate_Pct,Working_Poverty_Rate_Pct,Gender,Urban_Rural,Educational_Attainment,Sector_Employment_Pct,Average_Hours_Worked_Weekly,Time_Related_Underemployment_Pct,Methodology,Key_Context
0,2014,North Central,Abuja FCT,7.2,19.5,9.8,6.5,62.3,15.2,65.4,28.5,Male,Urban,Post-Secondary,18.5,42.0,12.3,Old,Pre-recession baseline; FCT government employm...
1,2014,North Central,Abuja FCT,8.1,21.3,11.2,7.2,58.7,18.5,68.2,32.1,Female,Urban,Secondary,15.2,38.5,14.5,Old,Gender gap evident in capital city
2,2014,North Central,Benue,8.5,20.8,10.5,7.8,65.2,16.8,72.5,35.8,Male,Rural,Primary,8.5,35.0,18.2,Old,Agrarian state seasonal employment
3,2014,North Central,Benue,9.2,22.1,12.8,8.5,60.5,19.2,75.8,40.2,Female,Rural,No Formal,5.2,30.0,20.5,Old,Rural women subsistence farming
4,2014,North East,Borno,8.1,21.3,11.5,7.2,58.2,20.5,70.2,38.5,Male,Urban,Secondary,15.8,40.5,15.2,Old,Early insurgency impact visible
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112,2022,North West,Kano,8.5,13.2,11.8,7.8,59.5,16.5,66.5,32.5,Male,Urban,Secondary,21.5,40.0,11.0,New,Rural working-hour threshold effect
113,2022,North West,Kano,10.2,15.5,13.8,9.5,53.0,19.8,71.5,38.0,Female,Urban,No Formal,7.5,31.0,13.5,New,Female informal work recognition
114,2022,South East,Abia,15.2,16.5,19.0,14.0,56.0,21.5,75.5,38.5,Male,Urban,Tertiary,24.0,39.0,14.0,New,Highest under new methodology South East
115,2022,South East,Abia,18.5,19.8,22.5,17.0,50.5,25.8,79.5,44.5,Female,Urban,Secondary,14.5,33.0,17.0,New,Abia structural unemployment persists


In [7]:
#Viewing the dataset
unemployment.head(10)# Note that date is not an integer so it has to be changed to time series.

,Year,Geopolitical_Zone,State_Example,Unemployment_Rate_Pct,Underemployment_Rate_Pct,Youth_Unemployment_15_24_Pct,Youth_Unemployment_25_34_Pct,Labor_Force_Participation_Rate_Pct,NEET_Rate_Pct,Informal_Employment_Rate_Pct,Working_Poverty_Rate_Pct,Gender,Urban_Rural,Educational_Attainment,Sector_Employment_Pct,Average_Hours_Worked_Weekly,Time_Related_Underemployment_Pct,Methodology,Key_Context
0,2014,North Central,Abuja FCT,7.2,19.5,9.8,6.5,62.3,15.2,65.4,28.5,Male,Urban,Post-Secondary,18.5,42.0,12.3,Old,Pre-recession baseline; FCT government employm...
1,2014,North Central,Abuja FCT,8.1,21.3,11.2,7.2,58.7,18.5,68.2,32.1,Female,Urban,Secondary,15.2,38.5,14.5,Old,Gender gap evident in capital city
2,2014,North Central,Benue,8.5,20.8,10.5,7.8,65.2,16.8,72.5,35.8,Male,Rural,Primary,8.5,35.0,18.2,Old,Agrarian state seasonal employment
3,2014,North Central,Benue,9.2,22.1,12.8,8.5,60.5,19.2,75.8,40.2,Female,Rural,No Formal,5.2,30.0,20.5,Old,Rural women subsistence farming
4,2014,North East,Borno,8.1,21.3,11.5,7.2,58.2,20.5,70.2,38.5,Male,Urban,Secondary,15.8,40.5,15.2,Old,Early insurgency impact visible
5,2014,North East,Borno,9.5,24.2,14.2,8.8,52.5,25.8,74.5,42.8,Female,Rural,No Formal,3.5,28.0,22.8,Old,Conflict-affected women limited opportunities
6,2014,North East,Adamawa,7.8,20.5,10.8,6.8,60.2,17.5,68.8,32.5,Male,Urban,Tertiary,20.2,43.0,13.5,Old,University town Yola employment
7,2014,North East,Adamawa,8.5,22.8,12.5,7.5,56.5,20.2,72.5,36.8,Female,Rural,Primary,8.2,32.0,18.5,Old,Agricultural processing limited
8,2014,North West,Kano,6.8,18.2,9.2,5.8,64.5,14.2,62.5,25.8,Male,Urban,Secondary,22.5,45.0,11.2,Old,Commercial center manufacturing
9,2014,North West,Kano,7.5,20.5,10.8,6.5,58.2,18.5,68.2,30.2,Female,Urban,No Formal,8.5,35.0,16.5,Old,Purdah limits female participation


In [9]:
unemployment.info()

<class 'pandas.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Year                                117 non-null    int64  
 1   Geopolitical_Zone                   117 non-null    str    
 2   State_Example                       117 non-null    str    
 3   Unemployment_Rate_Pct               117 non-null    float64
 4   Underemployment_Rate_Pct            117 non-null    float64
 5   Youth_Unemployment_15_24_Pct        117 non-null    float64
 6   Youth_Unemployment_25_34_Pct        117 non-null    float64
 7   Labor_Force_Participation_Rate_Pct  117 non-null    float64
 8   NEET_Rate_Pct                       117 non-null    float64
 9   Informal_Employment_Rate_Pct        117 non-null    float64
 10  Working_Poverty_Rate_Pct            117 non-null    float64
 11  Gender                              117 non-null    str 

In [10]:
unemployment['Key_Context']#Drop it later in feature engineering.

0      Pre-recession baseline; FCT government employm...
1                     Gender gap evident in capital city
2                     Agrarian state seasonal employment
3                        Rural women subsistence farming
4                        Early insurgency impact visible
                             ...                        
112                  Rural working-hour threshold effect
113                     Female informal work recognition
114             Highest under new methodology South East
115                Abia structural unemployment persists
116                                                  NaN
Name: Key_Context, Length: 117, dtype: str

In [11]:
unemployment.describe()

,Year,Unemployment_Rate_Pct,Underemployment_Rate_Pct,Youth_Unemployment_15_24_Pct,Youth_Unemployment_25_34_Pct,Labor_Force_Participation_Rate_Pct,NEET_Rate_Pct,Informal_Employment_Rate_Pct,Working_Poverty_Rate_Pct,Sector_Employment_Pct,Average_Hours_Worked_Weekly,Time_Related_Underemployment_Pct
count,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000,117.000000
mean,2017.487179,16.219658,24.993162,20.070940,14.896581,56.135043,24.456410,71.407692,38.294017,15.765812,36.529915,18.992308
std,2.680116,8.206455,4.911663,8.586166,7.756576,5.093223,6.553196,5.513684,7.389348,7.609331,6.184063,4.398279
min,2014.000000,6.400000,13.200000,8.500000,5.500000,41.500000,12.500000,58.200000,22.500000,0.800000,16.000000,10.500000
25%,2015.000000,9.200000,21.500000,12.800000,8.500000,53.000000,19.500000,67.500000,32.500000,10.800000,34.000000,15.500000
50%,2017.000000,14.100000,25.100000,17.800000,12.800000,56.500000,23.500000,71.200000,37.800000,15.200000,37.000000,18.500000
75%,2020.000000,20.500000,28.500000,25.500000,18.800000,59.800000,29.000000,75.500000,43.200000,21.000000,41.000000,22.000000
max,2022.000000,38.500000,38.500000,42.500000,36.000000,66.500000,42.500000,83.500000,58.500000,32.500000,48.000000,32.500000


In [15]:
unemployment['Key_Context'] = unemployment['Key_Context'].fillna('Unknown')

In [16]:
unemployment.info()

<class 'pandas.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Year                                117 non-null    int64  
 1   Geopolitical_Zone                   117 non-null    str    
 2   State_Example                       117 non-null    str    
 3   Unemployment_Rate_Pct               117 non-null    float64
 4   Underemployment_Rate_Pct            117 non-null    float64
 5   Youth_Unemployment_15_24_Pct        117 non-null    float64
 6   Youth_Unemployment_25_34_Pct        117 non-null    float64
 7   Labor_Force_Participation_Rate_Pct  117 non-null    float64
 8   NEET_Rate_Pct                       117 non-null    float64
 9   Informal_Employment_Rate_Pct        117 non-null    float64
 10  Working_Poverty_Rate_Pct            117 non-null    float64
 11  Gender                              117 non-null    str 

In [16]:
unemployment['Year'] = pd.to_datetime('2014')  # Changing the year to date or time series, ask AI about this?

In [ ]:
#Feature Engineering:scaling(normalization) due to difference in the measurement of the average hour and rate percentage. Must I remove outliers before scaling?

In [4]:
#EDA:univariant, multivariant, distribution and correlation. We returned back to feature engineering to create dummies due to high correlation in the dataset.



Geopolitical_Zone
North Central    20
North East       20
North West       20
South East       20
South South      19
South West       18
Name: count, dtype: int64

In [ ]:
#Correlation Matrix? how do i know which feature to select from this.


In [ ]:
#Machine Learning
